# 実験Ⅳ－２: example02_01 分析ノートブック
このノートブックはフォルダ内の既存コードを基に作成した解析ノートです。
各セルの先頭に「課題1」「課題2」…と記載し、該当する課題が分かるようにしています。

In [ ]:
# 共通準備 (imports 等)
%matplotlib inline
import warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import wave
import statsmodels.api as sm
from statsmodels.tsa import stattools
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.ar_model import AutoReg
sns.set(font_scale=1.0)
plt.rcParams['figure.dpi'] = 120

## 課題1: サンプルデータの読み込み
`data/practice02_01_2026.csv` が存在すればCSVを優先で読み込み、なければ WAV (`example02_01_2026.wav`) を読み込みます。

In [ ]:
# 課題1 実装: データ読み込み

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    # 想定列: t, y  なければ最初の数値列を y として扱う
    if 't' not in df.columns or 'y' not in df.columns:
        numcols = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numcols) >= 1:
            df = pd.DataFrame({'t': np.arange(len(df)), 'y': df[numcols[0]].values})
elif os.path.exists(wav_path):
    wf = wave.open(wav_path, mode='rb')
    buf = wf.readframes(-1)
    ch = wf.getnchannels()
    fs = wf.getframerate()
    wf.close()
    y = np.frombuffer(buf, dtype='int16')
    if ch == 2:
        y = y.reshape(-1, 2).mean(axis=1).astype('int16')
    t = np.arange(len(y)) / float(fs)
    df = pd.DataFrame({'t': t, 'y': y})
else:
    raise FileNotFoundError(f'入力データが見つかりません: {csv_path} または {wav_path}')
# 表示: 先頭数行
df.head()

## 課題2: 平均 μ と分散 γ0 の推定（原系列および階差系列）

In [ ]:
# 課題2 実装: 平均・分散の計算
y = df['y'].astype(float).reset_index(drop=True)
mu_y = y.mean()
gamma0_y_unbiased = y.var(ddof=1)
gamma0_y_mle = y.var(ddof=0)
# 階差系列
y_diff = y.diff().dropna()
mu_diff = y_diff.mean()
gamma0_diff = y_diff.var(ddof=1)
print(f'原系列: 平均 μ = {mu_y:.6f}, 分散 (不偏) = {gamma0_y_unbiased:.4f}')
print(f'階差系列: 平均 μ = {mu_diff:.6f}, 分散 (不偏) = {gamma0_diff:.4f}')

## 課題3: 単位根検定 (ADF) — 原系列と階差系列で検定を行う

In [ ]:
# 課題3 実装: ADF 検定
def run_adf(series, name='series'):
    try:
        res = adfuller(series.dropna())
        print(f'ADF {name}: stat={res[0]:.4f}, pvalue={res[1]:.4g}, usedlag={res[2]}')
    except Exception as e:
        print('ADF error for', name, e)
run_adf(y, '原系列 y')
run_adf(y_diff, '階差系列 Δy')

## 課題4: 自己共分散 γ_k・自己相関 ρ_k の推定とコレログラム (ACF/PACF)

In [ ]:
# 課題4 実装: ACF / PACF の計算とプロット
nlags = min(40, max(10, len(y_diff)//4))
lags = np.arange(nlags+1)
gamma_pd = np.array([y_diff.cov(y_diff.shift(k)) for k in lags])
rho_pd = np.array([y_diff.corr(y_diff.shift(k)) for k in lags])
try:
    acf_vals = stattools.acf(y_diff, nlags=nlags)
    pacf_vals = stattools.pacf(y_diff, nlags=nlags)
except Exception:
    acf_vals = rho_pd
    pacf_vals = np.zeros_like(acf_vals)
# プロット
fig, axes = plt.subplots(2, 1, figsize=(10,6))
axes[0].stem(lags, gamma_pd, basefmt='k-')
axes[0].set_title('自己共分散 γ_k (階差系列)')
axes[0].set_xlabel('ラグ k')
axes[1].stem(lags, acf_vals, basefmt='k-')
axes[1].set_title('自己相関 ρ_k (階差系列) / ACF')
axes[1].set_xlabel('ラグ k')
plt.tight_layout()
plt.show()

## 課題5: AR モデル推定・モデル選択 (AIC)・簡易予測

In [ ]:
# 課題5 実装: AutoReg を用いた AR(p) の AIC による選択と予測
y_for_ar = y_diff.dropna()
best_aic = np.inf
best_lag = None
best_model = None
max_p = min(12, max(1, len(y_for_ar)//2))
for p in range(1, max_p+1):
    try:
        model = AutoReg(y_for_ar, lags=p, old_names=False).fit()
        aic = model.aic
        if aic < best_aic:
            best_aic = aic
            best_lag = p
            best_model = model
    except Exception:
        continue
print(f'選択されたラグ数 p = {best_lag}, AIC = {best_aic:.3f}')
# 予測 (短期)
if best_model is not None:
    h = min(50, max(5, len(y_for_ar)//10))
    pred = best_model.predict(start=len(y_for_ar), end=len(y_for_ar)+h-1)
    plt.figure(figsize=(10,3))
    plt.plot(y_for_ar.index, y_for_ar.values, label='階差系列 (学習)')
    plt.plot(np.arange(len(y_for_ar), len(y_for_ar)+h), pred, label=f'予測 h={h}', linestyle='--')
    plt.legend()
    plt.title(f'AR({best_lag}) による階差系列の予測')
    plt.show()
else:
    print('AR モデルの推定に失敗しました。データ量を確認してください。')

## 保存 / 提出用まとめセル
解析結果の要約をCSVとして保存します。必要に応じて図の保存も追加してください。

In [ ]:
out_dir = 'example02_01_analysis_results'
os.makedirs(out_dir, exist_ok=True)
summary = {
    'n_samples': int(len(y)),
    'mean_y': float(mu_y),
    'var_y_unbiased': float(gamma0_y_unbiased),
    'mean_diff': float(mu_diff),
    'var_diff': float(gamma0_diff),
    'selected_ar_lag': int(best_lag) if best_lag is not None else None,
    'selected_ar_aic': float(best_aic) if best_lag is not None else None,
}
pd.DataFrame([summary]).to_csv(os.path.join(out_dir, 'summary.csv'), index=False)
print('summary.csv を出力しました: ', os.path.join(out_dir, 'summary.csv'))